# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [8]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [9]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [10]:
#your code here
# Exploración rápida antes de limpiar
print("Shape original:", spaceship.shape)
print("\nNulos por columna:")
print(spaceship.isnull().sum())

Shape original: (8693, 14)

Nulos por columna:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [11]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [ ]:
#Limpieza y preparación de datos
# Eliminamos nulos — son menos del 3% del total, no afecta significativamente
spaceship = spaceship.dropna().copy()
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship.drop(columns=["PassengerId", "Name"], inplace=True)
spaceship = pd.get_dummies(spaceship, drop_first=True, dtype=int)

X = spaceship.drop(columns=["Transported"]) # 19 columnas sin el target
y = spaceship["Transported"] # solo target

In [15]:
spaceship.shape

(6606, 20)

In [16]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (5284, 19)
Test: (1322, 19)


In [17]:
# Feature scaling 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Aunque los modelos basados en árboles como Gradient Boosting no necesitan escalado, lo aplicamos por consistencia con los labs anteriores para que la comparación sea justa.

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [18]:
#your code here
# Mejor modelo — Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

# Iniciar modelo
gb = GradientBoostingClassifier(random_state=0)

# Training
gb.fit(X_train_scaled, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


- Evaluate your model

In [19]:
#your code here
pred = gb.predict(X_test_scaled)
print("Train accuracy:", gb.score(X_train_scaled, y_train))
print("Test accuracy:", gb.score(X_test_scaled, y_test))
print(classification_report(y_test, pred))

Train accuracy: 0.8236184708554126
Test accuracy: 0.7866868381240545
              precision    recall  f1-score   support

       False       0.82      0.74      0.78       661
        True       0.76      0.83      0.80       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [31]:
#your code here
from sklearn.model_selection import GridSearchCV

grid = {
"n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_leaf_nodes": [10, 50, None]
}

Se ha tomado como referencia los valores usados en el notebook de clase y los hemos adaptado al problema de clasificación:
- n_estimators: probamos pocos árboles (50), el valor por defecto (100) y más (200)
- max_depth: árboles poco profundos (3), moderados (5) y más complejos (7)
- learning_rate: aprendizaje lento (0.01), estándar (0.1) y más rápido (0.2)
- max_leaf_nodes: árboles restringidos (10), moderados (50) y sin límite (None)

- Run Grid Search

In [32]:
model = GridSearchCV(estimator=gb, param_grid=grid, cv=5, n_jobs=-1)
model.fit(X_train_scaled, y_train)

,estimator,GradientBoost...andom_state=0)
,param_grid,"{'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'max_leaf_nodes': [10, 50, ...], 'n_estimators': [50, 100, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [33]:
# Mejores parámetros encontrados
print("Mejores parámetros:", model.best_params_)

Mejores parámetros: {'learning_rate': 0.2, 'max_depth': 7, 'max_leaf_nodes': 10, 'n_estimators': 50}


Se usa cv=5 porque es el estándar más habitual en Machine Learning, ofrece un buen equilibrio entre fiabilidad del resultado y tiempo de cómputo. 
Con menos folds (cv=3) la evaluación es menos estable; con más folds (cv=10) tarda significativamente más sin una mejora proporcional en la fiabilidad.

In [34]:
# Recuperar mejor modelo
best_model = model.best_estimator_

- Evaluate your model

In [35]:
pred = best_model.predict(X_test_scaled)
print("Train accuracy:", best_model.score(X_train_scaled, y_train))
print("Test accuracy:", best_model.score(X_test_scaled, y_test))
print(classification_report(y_test, pred))

Train accuracy: 0.838001514004542
Test accuracy: 0.7934947049924357
              precision    recall  f1-score   support

       False       0.81      0.76      0.79       661
        True       0.77      0.83      0.80       661

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



In [36]:
# Tabla comparativa final
resultados = {}

resultados["GB sin tuning"] = {
    "train": round(gb.score(X_train_scaled, y_train) * 100, 1),
    "test": round(gb.score(X_test_scaled, y_test) * 100, 1)
}

resultados["GB con tuning"] = {
    "train": round(best_model.score(X_train_scaled, y_train) * 100, 1),
    "test": round(best_model.score(X_test_scaled, y_test) * 100, 1)
}

tabla = pd.DataFrame(resultados).T
tabla.columns = ["Train", "Test"]
tabla["Overfitting"] = (tabla["Train"] - tabla["Test"]).round(1)
tabla

,Train,Test,Overfitting
GB sin tuning,82.4,78.7,3.7
GB con tuning,83.8,79.3,4.5


**Hyperparameter Tuning**

En el lab anterior de Ensemble Methods, Gradient Boosting fue el modelo con mejor equilibrio entre porcentaje de aciertos en datos nuevos (78.7%) y overfitting (3.7pts).
Por eso lo elegimos como modelo base para aplicar el tuning.

El objetivo es ver si ajustando sus hiperparámetros podemos mejorar ese 78.7%.

Grid Search prueba todas las combinaciones posibles de hiperparámetros. Cada combinación se evalúa con Cross Validation (cv=5) es decir, los datos se dividen en 5 partes y 
cada combinación se entrena y evalúa 5 veces con divisiones distintas, dando un resultado más fiable que un único train/test split.

**Conclusión**

Grid Search encontró los mejores parámetros dentro del rango probado:
- learning_rate: 0.2 → qué tan rápido aprende el modelo de sus errores en cada iteración
- max_depth: 7 → profundidad máxima de cada árbol, es decir, cuántas preguntas puede hacer
- max_leaf_nodes: 10 → número máximo de nodos hoja (predicciones finales) en cada árbol
- n_estimators: 50 → número de árboles encadenados que forman el modelo

El modelo con tuning mejora el test accuracy de 78.7% a 79.3% manteniendo un overfitting razonable (4.5pts).

Los hiperparámetros encontrados son los mejores de los probados  y no necesariamente los óptimos absolutos. Con un grid más amplio o usando Random Search podríamos seguir explorando mejores combinaciones.